# Scoring Linguistic Features: Original vs. Modified Statements

This notebook builds the linguistic feature table used in the manuscript: 26 DeCLaRatiVE deception features plus four part-of-speech features (adjectives, nouns, verbs, adverbs), computed per original and modified statement from `data/combined_long_liwc.csv`.

It then compares original and modified statements on every feature — overall and separately by attack modality (human / generative language model) — using permutation tests with Holm correction and effect sizes (Cohen's *d*, CLES).

Outputs:
- `output/combined_long_lingfeat.csv`: compact feature table (one row per text)
- `output/combined_long_lingfeat_linked.csv`: features plus identifiers and text (input to the classifier notebook)

In [1]:
from pathlib import Path
from collections import Counter

import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
import textstat
import pingouin as pg

from scipy.stats import permutation_test
from statsmodels.stats.multitest import multipletests

In [ ]:
# Ensure the required spaCy transformer model is available in the environment.
if not spacy.util.is_package("en_core_web_trf"):
    raise RuntimeError(
        "Missing spaCy model 'en_core_web_trf'. Install it once in your environment with: "
        "python -m spacy download en_core_web_trf"
    )

print("spaCy model 'en_core_web_trf' is available.")

In [ ]:
ROOT = Path('..').resolve()
DATA_PATH = ROOT / 'data' / 'combined_long_liwc.csv'
CONCR_PATH = ROOT / 'linguisticFeatures' / 'Brysbaert concreteness scoring.xlsx'
OUTPUT_PATH = ROOT / 'output' / 'combined_long_lingfeat.csv'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH, encoding='utf-8')
required_cols = {'text', 'text_type', 'WC'}
missing = sorted(required_cols - set(df.columns))
if missing:
    raise ValueError(f'Missing required columns: {missing}')

# Keep variable naming aligned with the original notebook flow.
df_h = df[df['text_type'].astype(str).str.lower() == 'original']
df_d = df[df['text_type'].astype(str).str.lower() == 'rewrite']

print(f'Rows: {len(df):,}')
print(f'Original rows: {len(df_h):,}')
print(f'Rewrite rows: {len(df_d):,}')

In [ ]:
def load_spacy_model():
    try:
        return spacy.load('en_core_web_trf')
    except OSError as exc:
        raise RuntimeError(
            "Missing spaCy model 'en_core_web_trf'. Install it with: "
            "python -m spacy download en_core_web_trf"
        ) from exc

nlp = load_spacy_model()
docs = [doc for doc in nlp.pipe(df['text'].fillna(''), batch_size=128)]

In [ ]:
def tokenize(doc):
    tokens = [token.text for token in doc if not (token.is_punct or token.is_space)]
    return ' '.join(tokens).lower()

def preprocessed_text(doc):
    clean_tokens = [
        token.lemma_
        for token in doc
        if not (token.is_punct or token.is_space or token.is_stop)
    ]
    return ' '.join(clean_tokens).lower()

tokens = [tokenize(doc) for doc in docs]
clean_text = [preprocessed_text(doc) for doc in docs]

df['tokens'] = tokens
df['clean_text'] = clean_text

word_counts = [len(text.split()) for text in df['tokens']]
df['word_counts'] = word_counts

# Recreate group splits now that word_counts is added to df
df_h = df[df['text_type'].astype(str).str.lower() == 'original']
df_d = df[df['text_type'].astype(str).str.lower() == 'rewrite']

df[['text_type', 'word_counts', 'tokens', 'clean_text']].head()

## Word Count Exploration

Quick checks on text length distribution before feature extraction.

In [ ]:
word_count_summary = (
    df.groupby("text_type", dropna=False)["word_counts"]
    .agg(["count", "min", "max", "mean", "std", "median"])
    .round(2)
)
display(word_count_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, group in df.groupby("text_type"):
    axes[0].hist(group["word_counts"], bins=20, alpha=0.6, label=str(label))
axes[0].set_title("Word Count Histogram by text_type")
axes[0].set_xlabel("word_counts")
axes[0].set_ylabel("Frequency")
axes[0].legend()

sns.boxplot(data=df, x="text_type", y="word_counts", ax=axes[1])
axes[1].set_title("Word Count Boxplot by text_type")

plt.tight_layout()
plt.show()

## Textstat Features

Compute sentence- and readability-based text statistics.

In [ ]:
text_stats_rows = []
for text in df['text'].fillna(''):
    num_sentences = textstat.sentence_count(text)
    num_words = textstat.lexicon_count(text)
    num_syllables = textstat.syllable_count(text)
    avg_syllables_per_word = (num_syllables / num_words) if num_words else 0
    fk_grade = textstat.flesch_kincaid_grade(text)
    fk_read = textstat.flesch_reading_ease(text)
    text_stats_rows.append({
        'num_sentences': num_sentences,
        'num_syllables': num_syllables,
        'avg_syllables_per_word': avg_syllables_per_word,
        'fk_grade': fk_grade,
        'fk_read': fk_read
    })

text_stats_df = pd.DataFrame(text_stats_rows)
text_stats_df.head()

## LIWC Composite Features

Build the LIWC-based composite variables used in the feature table.

In [ ]:
liwc_final = df[[
    'Analytic', 'Authentic', 'Tone', 'tone_pos', 'tone_neg',
    'focuspast', 'focuspresent', 'focusfuture', 'Cognition', 'memory'
]].copy()

liwc_final.loc[:, 'Self-reference'] = df['i'] + df['we']
liwc_final.loc[:, 'Other-reference'] = df['shehe'] + df['they'] + df['you']
liwc_final.loc[:, 'Contextual Embedding'] = df['space'] + df['motion'] + df['time']
liwc_final.loc[:, 'Perceptual Details'] = df['attention'] + df['visual'] + df['auditory'] + df['feeling']
liwc_final.loc[:, 'Reality Monitoring'] = (
    liwc_final['Perceptual Details'] + liwc_final['Contextual Embedding'] + df['Affect'] - df['Cognition']
)

liwc_final.head()

In [ ]:
liwc_final.describe()

## Named Entity Recognition (NER) Features

Extract NER counts and grouped NER constructs.

In [ ]:
# NER extraction and grouped NER constructs
ner = []
for doc in docs:
    entities = set((entity.text, entity.label_) for entity in doc.ents)
    entity_counts = {}
    for entity, label in entities:
        if label not in entity_counts:
            entity_counts[label] = {entity: 1}
        elif entity not in entity_counts[label]:
            entity_counts[label][entity] = 1
        else:
            entity_counts[label][entity] += 1
    ner.append(entity_counts)

ner_df = pd.DataFrame.from_records([
    {col: sum(inner.values()) for col, inner in d.items()} for d in ner
]).fillna(0)
ner_df = ner_df.div(df['word_counts'].replace(0, 1), axis=0) * 100
ner_df['text_type'] = df['text_type'].values

def safe_col(frame, col):
    if col in frame.columns:
        return frame[col]
    return pd.Series(0.0, index=frame.index)

new_ner = pd.DataFrame(index=ner_df.index)
new_ner['Temporal details'] = safe_col(ner_df, 'DATE') + safe_col(ner_df, 'TIME') + safe_col(ner_df, 'EVENT')
new_ner['Spatial details'] = safe_col(ner_df, 'GPE') + safe_col(ner_df, 'LOC') + safe_col(ner_df, 'FAC')
new_ner['People'] = safe_col(ner_df, 'PERSON')
new_ner['Quantity details'] = (
    safe_col(ner_df, 'PERCENT') + safe_col(ner_df, 'MONEY') + safe_col(ner_df, 'QUANTITY')
    + safe_col(ner_df, 'CARDINAL') + safe_col(ner_df, 'ORDINAL')
)

new_ner.head()

In [ ]:
# NER diagnostics
print("Missing values per NER feature:")
display(new_ner.isna().sum())

print("NER feature summary statistics:")
display(new_ner.describe())

## Concreteness 

Compute average concreteness scores.

In [ ]:
# Concreteness
concr_df = pd.read_excel(CONCR_PATH)

def get_avg_concr_score(text):
    tokens = set(str(text).split())
    matches = concr_df['Word'].str.lower().isin(tokens)
    scores = concr_df.loc[matches, 'Conc.M'].values
    return float(scores.mean()) if len(scores) > 0 else 0.0

df['concr_score'] = df['clean_text'].apply(get_avg_concr_score)

df[['concr_score']].head()

## POS Features

Compute POS-tag percentages (adjectives, nouns, verbs, adverbs) from the parsed spaCy docs.

In [ ]:
# POS-tag percentages requested: adjectives, nouns, verbs, adverbs.
pos_rows = []
for doc, wc in zip(docs, df['word_counts'].replace(0, 1)):
    tags = Counter(token.pos_ for token in doc if not (token.is_punct or token.is_space))
    pos_rows.append({
        'adj_pct': tags.get('ADJ', 0) / wc * 100,
        'noun_pct': tags.get('NOUN', 0) / wc * 100,
        'verb_pct': tags.get('VERB', 0) / wc * 100,
        'adverb_pct': tags.get('ADV', 0) / wc * 100
    })
pos_tags_df = pd.DataFrame(pos_rows)

pos_tags_df.head()

## Feature Merge and Export

Merge all feature blocks into a final table and save compact plus linked outputs.

In [ ]:
# Final scoring dataframe (same merge logic, adapted label and with POS tags included)
labels = pd.Series(df['text_type']).reset_index(drop=True)
word_counts_series = pd.Series(df['word_counts']).reset_index(drop=True)
concr_score = df['concr_score'].reset_index(drop=True)

scoring_df = pd.concat([
    labels.rename('text_type'),
    word_counts_series.rename('word_counts'),
    text_stats_df.reset_index(drop=True),
    liwc_final.reset_index(drop=True),
    concr_score,
    new_ner.reset_index(drop=True),
    pos_tags_df.reset_index(drop=True)
], axis=1)

# Keep a compact feature table.
scoring_df.to_csv(OUTPUT_PATH, index=False)
print(f'Saved: {OUTPUT_PATH}')
print(f'Shape: {scoring_df.shape}')

# Also save a linked table that keeps identifiers/text from the original CSV.
link_cols = [
    col for col in [
        'rewrite_id', 'statement_id', 'topic_label', 'attack_modality',
        'original_veracity', 'target_veracity', 'original_label', 'rewrite_label',
        'asr', 'text_type', 'text'
    ] if col in df.columns
]

scoring_with_keys = pd.concat([
    df[link_cols].reset_index(drop=True),
    scoring_df.drop(columns=['text_type'], errors='ignore').reset_index(drop=True)
], axis=1)

linked_output_path = OUTPUT_PATH.with_name('combined_long_lingfeat_linked.csv')
scoring_with_keys.to_csv(linked_output_path, index=False)
print(f'Saved: {linked_output_path}')
print(f'Linked shape: {scoring_with_keys.shape}')

scoring_with_keys.head()

## Extended Step-by-Step Checks


In [ ]:
def num_of_words_stat(word_counts_vec):
    min_words = word_counts_vec.min()
    max_words = word_counts_vec.max()
    mean_words = word_counts_vec.mean()
    std_words = word_counts_vec.std()

    print(f"Minimum number of words: {min_words}")
    print(f"Maximum number of words: {max_words}")
    print(f"Average number of words: {mean_words:.2f}")
    print(f"Standard deviation number of words: {std_words:.2f}")

    return min_words, max_words, mean_words, std_words

print("All texts:")
_ = num_of_words_stat(df["word_counts"])

print("\nOriginal texts:")
_ = num_of_words_stat(df_h["word_counts"])

print("\nRewrite texts:")
_ = num_of_words_stat(df_d["word_counts"])

In [ ]:
# Distribution checks for word counts
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.hist(df["word_counts"], bins=20)
ax1.set_title("Histogram: word_counts")
ax2.boxplot(df["word_counts"])
ax2.set_title("Boxplot: word_counts")
plt.tight_layout()
plt.show()

# Potential long-text outliers
avg_wc = df["word_counts"].mean()
std_wc = df["word_counts"].std()
df[df["word_counts"] > avg_wc + 2.5 * std_wc].sort_values("word_counts", ascending=False).head(10)

In [ ]:
# Jaccard index between original and rewrite vocabularies
set_original = set(df_h["clean_text"].str.split().explode())
set_rewrite = set(df_d["clean_text"].str.split().explode())

intersection = set_original.intersection(set_rewrite)
union = set_original.union(set_rewrite)
jaccard_index = len(intersection) / len(union) if union else 0.0
print(f"Jaccard index (original vs rewrite): {jaccard_index:.3f}")

In [ ]:
# Jaccard index per attack modality
for mod in sorted(df['attack_modality'].dropna().unique()):
    mod_df = df[df['attack_modality'] == mod]
    set_orig_mod = set(
        mod_df[mod_df['text_type'].str.lower() == 'original']['clean_text']
        .str.split().explode()
    )
    set_rew_mod = set(
        mod_df[mod_df['text_type'].str.lower() == 'rewrite']['clean_text']
        .str.split().explode()
    )
    inter = set_orig_mod & set_rew_mod
    uni   = set_orig_mod | set_rew_mod
    ji = len(inter) / len(uni) if uni else 0.0
    print(f"Jaccard index ({mod}): {ji:.3f}  "
          f"(vocab orig={len(set_orig_mod):,}, rewrite={len(set_rew_mod):,}, "
          f"shared={len(inter):,})")


In [ ]:
# Split final feature table by group 
scoring_df_o = scoring_df[scoring_df["text_type"] == "original"].drop("text_type", axis=1)
scoring_df_r = scoring_df[scoring_df["text_type"] == "rewrite"].drop("text_type", axis=1)

print(f"Original feature rows: {len(scoring_df_o):,}")
print(f"Rewrite feature rows: {len(scoring_df_r):,}")
print(f"Feature count: {scoring_df_o.shape[1]}")

In [ ]:
# Quick histogram overlays per feature (visual check)
for feature in scoring_df_o.columns:
    plt.figure(figsize=(7, 4))
    scoring_df_o[feature].plot.hist(alpha=0.5, bins=20, label="original")
    scoring_df_r[feature].plot.hist(alpha=0.5, bins=20, label="rewrite")
    plt.xlabel(feature)
    plt.ylabel("Frequency")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Boxplot comparison for all features (faceted for readability)
melted = scoring_df.melt(id_vars=["text_type"], var_name="feature", value_name="value")

g = sns.catplot(
    data=melted,
    x="text_type",
    y="value",
    col="feature",
    col_wrap=4,
    kind="box",
    sharey=False,
    height=3,
    aspect=1.1
)
g.set_titles("{col_name}")
g.set_axis_labels("group", "value")
plt.show()

In [ ]:
def statistic(x, y, axis):
    return np.mean(x, axis=axis) - np.mean(y, axis=axis)

def bootstrap_cles_ci(x, y, alpha=0.05, n_boot=1000):
    boot_cles = []
    x = np.asarray(x)
    y = np.asarray(y)
    for _ in range(n_boot):
        sample_x = np.random.choice(x, size=len(x), replace=True)
        sample_y = np.random.choice(y, size=len(y), replace=True)
        cles = pg.compute_effsize(sample_x, sample_y, eftype="CLES")
        boot_cles.append(cles)
    lower = np.percentile(boot_cles, 100 * (alpha / 2))
    upper = np.percentile(boot_cles, 100 * (1 - alpha / 2))
    return round(float(lower), 3), round(float(upper), 3)

In [ ]:
# Cohen's d chart for significant features
perm_df_selected = perm_df[perm_df["null_h_rejected"]].copy()
perm_df_sorted = perm_df_selected.sort_values("effect_size_cohens_d", ascending=True)

if perm_df_sorted.empty:
    print("No significant features after Holm correction.")
else:
    fig, ax = plt.subplots(figsize=(8, max(4, 0.25 * len(perm_df_sorted))))
    colors = ["steelblue" if stat > 0 else "salmon" for stat in perm_df_sorted["stat"]]
    ax.barh(perm_df_sorted["feature"], perm_df_sorted["effect_size_cohens_d"], color=colors)
    ax.set_xlabel("Cohen's d")
    ax.set_ylabel("Feature")
    ax.set_title("Effect sizes by feature (original - rewrite)")
    plt.tight_layout()
    plt.show()

In [ ]:
# CLES stacked chart for significant features
perm_df_selected = perm_df[perm_df["null_h_rejected"]].copy()
perm_df_sorted = perm_df_selected.sort_values("effect_size_cles", ascending=True)

if perm_df_sorted.empty:
    print("No significant features after Holm correction.")
else:
    perm_df_sorted["one_minus_cles"] = 1 - perm_df_sorted["effect_size_cles"]
    fig, ax = plt.subplots(figsize=(9, max(4, 0.3 * len(perm_df_sorted))))
    ax.barh(
        perm_df_sorted["feature"],
        perm_df_sorted["effect_size_cles"],
        color="skyblue",
        label="original"
    )
    ax.barh(
        perm_df_sorted["feature"],
        perm_df_sorted["one_minus_cles"],
        left=perm_df_sorted["effect_size_cles"],
        color="salmon",
        label="rewrite"
    )
    ax.set_xlabel("CLES")
    ax.set_ylabel("Feature")
    ax.set_title("CLES decomposition by feature")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

## Feature Analysis by Attack Modality

Perform statistical tests and effect size calculations for each attack modality separately.


In [ ]:
# Get unique attack modalities and run analysis for each
modalities = sorted(scoring_with_keys['attack_modality'].dropna().unique())
print(f"Attack modalities found: {modalities}")

# Store results for each modality
modality_results = {}

for modality in modalities:
    print(f"\n{'='*60}")
    print(f"Attack Modality: {modality}")
    print(f"{'='*60}")
    
    # Filter data for this modality
    mod_data = scoring_with_keys[scoring_with_keys['attack_modality'] == modality]
    mod_scoring = scoring_df[scoring_with_keys['attack_modality'] == modality]
    
    # Split by text_type
    mod_orig = mod_scoring[mod_scoring['text_type'] == 'original'].drop('text_type', axis=1)
    mod_rewr = mod_scoring[mod_scoring['text_type'] == 'rewrite'].drop('text_type', axis=1)
    
    print(f"Original texts: {len(mod_orig)}, Rewrite texts: {len(mod_rewr)}")
    
    if len(mod_orig) == 0 or len(mod_rewr) == 0:
        print("Skipping: insufficient data for this modality")
        continue
    
    # Run permutation tests and effect sizes
    stats = []
    p_values = []
    cles_scores = []
    cohens_d_scores = []
    
    for col in mod_orig.columns:
        x = mod_orig[col].to_numpy()
        y = mod_rewr[col].to_numpy()
        
        # Remove NaN values
        x_clean = x[~np.isnan(x)]
        y_clean = y[~np.isnan(y)]
        
        # Skip if insufficient data after NaN removal
        if len(x_clean) < 2 or len(y_clean) < 2:
            stats.append(np.nan)
            p_values.append(np.nan)
            cles_scores.append(np.nan)
            cohens_d_scores.append(np.nan)
            continue
        
        res = permutation_test((x_clean, y_clean), statistic, vectorized=True, n_resamples=10000)
        stats.append(float(res.statistic))
        p_values.append(float(res.pvalue))
        
        cles = pg.compute_effsize(x_clean, y_clean, eftype="CLES")
        cles_scores.append(float(cles))
        
        cohens_d = pg.compute_effsize(x_clean, y_clean, paired=False, eftype="cohen")
        cohens_d_scores.append(cohens_d)
    
    # Handle NaN values in p_values for multipletests
    p_values_array = np.array(p_values)
    valid_mask = ~np.isnan(p_values_array)
    
    null_h_rej = np.full(len(p_values), False)
    corrected_p_values = np.array(p_values)
    
    if valid_mask.sum() > 0:
        rej_valid, corr_valid, _, _ = multipletests(p_values_array[valid_mask], alpha=0.05, method="holm")
        null_h_rej[valid_mask] = rej_valid
        corrected_p_values[valid_mask] = corr_valid
    
    mod_results = pd.DataFrame({
        "feature": mod_orig.columns,
        "stat": stats,
        "p_value": p_values,
        "corrected_p_value": corrected_p_values,
        "effect_size_cles": cles_scores,
        "effect_size_cohens_d": cohens_d_scores,
        "null_h_rejected": null_h_rej
    })
    mod_results["significance"] = ["*" if x else "-" for x in mod_results["null_h_rejected"]]
    mod_results = mod_results.sort_values(["null_h_rejected", "effect_size_cles"], ascending=False)
    
    modality_results[modality] = mod_results
    
    print(f"\nSignificant features (Holm corrected p < 0.05): {mod_results['null_h_rejected'].sum()}")
    display(mod_results.head(30))


In [ ]:
# Cohen's d comparison chart for each modality
for modality in modalities:
    if modality not in modality_results:
        continue
    
    mod_df = modality_results[modality]
    mod_df_sig = mod_df[mod_df['null_h_rejected']].copy()
    
    if mod_df_sig.empty:
        print(f"\n{modality}: No significant features after Holm correction.")
        continue
    
    mod_df_sorted = mod_df_sig.sort_values('effect_size_cohens_d', ascending=True)
    
    fig, ax = plt.subplots(figsize=(9, max(4, 0.3 * len(mod_df_sorted))))
    colors = ["steelblue" if stat > 0 else "salmon" for stat in mod_df_sorted["stat"]]
    ax.barh(mod_df_sorted["feature"], mod_df_sorted["effect_size_cohens_d"], color=colors)
    ax.set_xlabel("Cohen's d")
    ax.set_ylabel("Feature")
    ax.set_title(f"Effect sizes by feature ({modality}): original - rewrite")
    plt.tight_layout()
    plt.show()


In [ ]:
# Summary comparison table: which features are significant per modality
all_features = set()
for mod_df in modality_results.values():
    all_features.update(mod_df[mod_df['null_h_rejected']]['feature'].tolist())

summary_table = pd.DataFrame(index=sorted(all_features))
for modality in modalities:
    if modality not in modality_results:
        continue
    mod_df = modality_results[modality]
    sig_features = set(mod_df[mod_df['null_h_rejected']]['feature'])
    summary_table[f'{modality}_significant'] = summary_table.index.isin(sig_features).astype(int)
    
    # Add Cohen's d values
    for idx in summary_table.index:
        if idx in mod_df['feature'].values:
            d_val = mod_df[mod_df['feature'] == idx]['effect_size_cohens_d'].values[0]
            summary_table.loc[idx, f'{modality}_cohens_d'] = round(d_val, 3)

print("\nSignificant Features by Attack Modality:")
display(summary_table)


In [ ]:
# Boxplots showing distributions by text_type within each modality (selected significant features)
for modality in modalities:
    if modality not in modality_results:
        continue
    
    mod_df = modality_results[modality]
    sig_features = mod_df[mod_df['null_h_rejected']]['feature'].tolist()
    
    if not sig_features:
        print(f"\n{modality}: No significant features to visualize.")
        continue
    
    # Get data for this modality
    mod_scoring_data = scoring_with_keys[scoring_with_keys['attack_modality'] == modality]
    mod_scoring_all = scoring_df[scoring_with_keys['attack_modality'] == modality]
    
    # Melt for plotting (limit to 8 significant features for readability)
    features_to_plot = sig_features[:8]
    plot_cols = ['text_type'] + features_to_plot
    plot_data = mod_scoring_all[plot_cols].copy()
    
    melted = plot_data.melt(id_vars=['text_type'], var_name='feature', value_name='value')
    
    g = sns.catplot(
        data=melted,
        x='text_type',
        y='value',
        col='feature',
        col_wrap=4,
        kind='box',
        sharey=False,
        height=3,
        aspect=1.1
    )
    g.set_titles("{col_name}")
    g.set_axis_labels("group", "value")
    fig = g.fig
    fig.suptitle(f"Significant Features by Text Type: {modality}", y=1.00)
    plt.tight_layout()
    plt.show()


In [ ]:
# Distribution overlap visualizations for each feature within each modality
for modality in modalities:
    if modality not in modality_results:
        continue
    
    # Get data for this modality
    mod_scoring_all = scoring_df[scoring_with_keys['attack_modality'] == modality]
    
    # Get all features (not just significant)
    features_list = [col for col in mod_scoring_all.columns if col != 'text_type']
    
    # Create violin plots for all features
    plot_cols = ['text_type'] + features_list
    plot_data = mod_scoring_all[plot_cols].copy()
    
    melted = plot_data.melt(id_vars=['text_type'], var_name='feature', value_name='value')
    
    g = sns.catplot(
        data=melted,
        x='text_type',
        y='value',
        col='feature',
        col_wrap=4,
        kind='violin',
        sharey=False,
        height=3,
        aspect=1.1
    )
    g.set_titles("{col_name}")
    g.set_axis_labels("text_type", "value")
    fig = g.fig
    fig.suptitle(f"Feature Distributions by Text Type: {modality} (All Features)", y=1.00)
    plt.tight_layout()
    plt.show()


## Feature Analysis by Modality × Original Veracity

For each combination of `attack_modality` (human / LLM) and `original_veracity` (Deceptive / Truthful), run permutation tests (Holm-corrected) and compute Cohen's d and CLES to compare **original** vs **rewrite** linguistic features.


In [ ]:
# ── Run permutation tests + effect sizes for each modality × veracity combo ──
veracities  = sorted(scoring_with_keys['original_veracity'].dropna().unique())
modalities  = sorted(scoring_with_keys['attack_modality'].dropna().unique())

# Nested dict: veracity_results[modality][veracity] = results DataFrame
veracity_results = {m: {} for m in modalities}

feature_cols = [c for c in scoring_with_keys.columns
                if c not in ['rewrite_id', 'statement_id', 'topic_label',
                              'attack_modality', 'original_veracity', 'target_veracity',
                              'original_label', 'rewrite_label', 'asr', 'text_type',
                              'text', 'word_counts']]

for modality in modalities:
    for veracity in veracities:
        mask = (
            (scoring_with_keys['attack_modality'] == modality) &
            (scoring_with_keys['original_veracity'] == veracity)
        )
        subset = scoring_with_keys[mask]

        sub_orig = subset[subset['text_type'] == 'original'][feature_cols]
        sub_rewr = subset[subset['text_type'] == 'rewrite'][feature_cols]

        label = f"{modality} / {veracity}"
        print(f"\n{'='*65}")
        print(f"Modality: {modality.upper()}  |  Veracity: {veracity}")
        print(f"  Original: {len(sub_orig)}  Rewrite: {len(sub_rewr)}")
        print(f"{'='*65}")

        if len(sub_orig) < 5 or len(sub_rewr) < 5:
            print("  Skipping: insufficient data.")
            continue

        stats_list, pvals, cles_list, d_list = [], [], [], []

        for col in feature_cols:
            x = sub_orig[col].to_numpy()
            y = sub_rewr[col].to_numpy()
            x_c = x[~np.isnan(x)]
            y_c = y[~np.isnan(y)]

            if len(x_c) < 2 or len(y_c) < 2:
                stats_list.append(np.nan)
                pvals.append(np.nan)
                cles_list.append(np.nan)
                d_list.append(np.nan)
                continue

            res = permutation_test((x_c, y_c), statistic, vectorized=True, n_resamples=10000)
            stats_list.append(float(res.statistic))
            pvals.append(float(res.pvalue))
            cles_list.append(float(pg.compute_effsize(x_c, y_c, eftype='CLES')))
            d_list.append(float(pg.compute_effsize(x_c, y_c, paired=False, eftype='cohen')))

        pvals_arr  = np.array(pvals)
        valid_mask = ~np.isnan(pvals_arr)
        null_rej   = np.full(len(pvals), False)
        corr_pvals = pvals_arr.copy()

        if valid_mask.sum() > 0:
            rej_v, corr_v, _, _ = multipletests(pvals_arr[valid_mask], alpha=0.05, method='holm')
            null_rej[valid_mask]   = rej_v
            corr_pvals[valid_mask] = corr_v

        res_df = pd.DataFrame({
            'feature':             feature_cols,
            'stat':                stats_list,
            'p_value':             pvals,
            'corrected_p_value':   corr_pvals,
            'effect_size_cles':    cles_list,
            'effect_size_cohens_d': d_list,
            'null_h_rejected':     null_rej,
        })
        res_df['significance'] = ['*' if v else '-' for v in res_df['null_h_rejected']]
        res_df = res_df.sort_values(['null_h_rejected', 'effect_size_cles'], ascending=False)

        veracity_results[modality][veracity] = res_df

        n_sig = res_df['null_h_rejected'].sum()
        print(f"  Significant features (Holm p < 0.05): {n_sig}")
        display(res_df.head(20))


In [ ]:
# ── Cohen's d bar charts for each modality × veracity with significant features ──
for modality in modalities:
    for veracity in veracities:
        if veracity not in veracity_results[modality]:
            continue
        res_df = veracity_results[modality][veracity]
        sig_df = res_df[res_df['null_h_rejected']].sort_values('effect_size_cohens_d', ascending=True)

        if sig_df.empty:
            print(f"\n{modality} / {veracity}: no significant features after Holm correction.")
            continue

        fig, ax = plt.subplots(figsize=(9, max(4, 0.3 * len(sig_df))))
        colors = ['steelblue' if s > 0 else 'salmon' for s in sig_df['stat']]
        ax.barh(sig_df['feature'], sig_df['effect_size_cohens_d'], color=colors, edgecolor='black')
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_xlabel("Cohen's d  (original − rewrite)")
        ax.set_title(f"Significant features — {modality.upper()} / {veracity}")
        ax.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.show()


In [ ]:
# ── Summary heatmap: Cohen's d across modality × veracity subgroups ───────────
# Collect all features that are significant in at least one subgroup
all_sig = set()
for modality in modalities:
    for veracity in veracities:
        if veracity in veracity_results[modality]:
            df_sig = veracity_results[modality][veracity]
            all_sig.update(df_sig[df_sig['null_h_rejected']]['feature'].tolist())

if all_sig:
    col_order = [f"{m} / {v}" for m in modalities for v in veracities
                 if v in veracity_results[m]]
    heat_df = pd.DataFrame(index=sorted(all_sig), columns=col_order, dtype=float)

    for modality in modalities:
        for veracity in veracities:
            col_label = f"{modality} / {veracity}"
            if veracity not in veracity_results[modality]:
                continue
            sub = veracity_results[modality][veracity].set_index('feature')
            for feat in all_sig:
                if feat in sub.index:
                    heat_df.loc[feat, col_label] = sub.loc[feat, 'effect_size_cohens_d']

    fig, ax = plt.subplots(figsize=(max(8, len(col_order) * 2), max(5, 0.3 * len(all_sig))))
    sns.heatmap(
        heat_df.astype(float),
        ax=ax,
        cmap='coolwarm',
        center=0,
        annot=True,
        fmt='.2f',
        linewidths=0.5,
        cbar_kws={'label': "Cohen's d"},
    )
    ax.set_title("Cohen's d: significant features across Modality × Veracity subgroups\n"
                 "(blank = not significant in that subgroup)")
    ax.set_xlabel("Subgroup")
    ax.set_ylabel("Feature")
    plt.tight_layout()
    plt.show()

    print("\nSignificance summary (1 = Holm-corrected p < 0.05, 0 = not significant):")
    sig_summary = pd.DataFrame(index=sorted(all_sig), columns=col_order, dtype=int)
    for modality in modalities:
        for veracity in veracities:
            col_label = f"{modality} / {veracity}"
            if veracity not in veracity_results[modality]:
                continue
            sub = veracity_results[modality][veracity].set_index('feature')
            for feat in all_sig:
                if feat in sub.index:
                    sig_summary.loc[feat, col_label] = int(sub.loc[feat, 'null_h_rejected'])
                else:
                    sig_summary.loc[feat, col_label] = 0
    display(sig_summary)
else:
    print("No significant features found across any subgroup.")


In [ ]:
# ── Boxplots: significant features per modality × veracity subgroup ───────────
for modality in modalities:
    for veracity in veracities:
        if veracity not in veracity_results[modality]:
            continue
        res_df = veracity_results[modality][veracity]
        sig_features = res_df[res_df['null_h_rejected']]['feature'].tolist()

        if not sig_features:
            print(f"\n{modality} / {veracity}: no significant features to plot.")
            continue

        mask = (
            (scoring_with_keys['attack_modality'] == modality) &
            (scoring_with_keys['original_veracity'] == veracity)
        )
        subset_plot = scoring_with_keys[mask]

        plot_cols = ['text_type'] + sig_features[:8]   # cap at 8 for readability
        melted_sub = subset_plot[plot_cols].melt(
            id_vars='text_type', var_name='feature', value_name='value'
        )

        g = sns.catplot(
            data=melted_sub,
            x='text_type', y='value',
            col='feature', col_wrap=4,
            kind='box', sharey=False,
            height=3, aspect=1.1,
        )
        g.set_titles('{col_name}')
        g.set_axis_labels('group', 'value')
        g.fig.suptitle(
            f"Significant features — {modality.upper()} / {veracity}  "
            f"(original vs rewrite)",
            y=1.01
        )
        plt.tight_layout()
        plt.show()
